In [ ]:
!pip3 install langchain langchain-community langchain-openai chromadb pypdf

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA



In [ ]:
# Define a chave da API da OpenAI como variável de ambiente
# (Nunca versionar essa chave em repositórios públicos)
os.environ["OPENAI_API_KEY"] = "COLE_SUA_CHAVE_AQUI"

In [6]:
docs_paths = [
    "documents/bula_sigmatriol.pdf",
    "documents/Bula-clortalidona-Profissional.pdf",
    "documents/Bula-Sigmatriol-Profissional-Consulta-Remedios.pdf",
    "documents/doenca_nodular_da_tireoide-diagnostico.pdf",
    "documents/fixare.pdf",
    "documents/Investigating_the_thyroid_nodule.pdf",
    "documents/Pediatric Thyroid Nodule.pdf",
    "documents/protocolo-clinico-e-diretrizes-terapeuticas-hipoparatireoidismo.pdf",
    "documents/The Oncologist - 2008 - Yeung - Management of the Solitary Thyroid Nodule.pdf",
    "documents/Thyroid Nodules-2.pdf",
    "documents/Thyroid Nodules.pdf"
]

documents = []

for path in docs_paths:

    # Create PDF loader
    loader = PyPDFLoader(path)

    # Load PDF content
    docs = loader.load()

    # Add name as metadata
    for doc in docs:
        doc.metadata["source"] = path.split("/")[-1].replace(".pdf", "")

    documents.extend(docs)

len(documents)

101

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,      
    chunk_overlap=150   
)

# Divide os documentos em chunks
chunks = text_splitter.split_documents(documents)

# Quantidade total de chunks gerados
len(chunks)

792

In [8]:
# Loop through each chunk to semantically classify its content
for chunk in chunks:

    # Normalize text to facilitate checks
    text = chunk.page_content.lower()

    # Hypoparathyroidism
    if "hipoparatireoidismo" in text or "paratireoide" in text or "hypoparathyroidism" in text or "parathyroid" in text or "pth" in text:
        chunk.metadata["topic"] = "hypoparathyroidism"

    # Thyroid nodules
    elif "nódulo" in text or "nodule" in text or "thyroid nodule" in text or "solitary thyroid" in text:
        chunk.metadata["topic"] = "thyroid_nodule"

    # Thyroid cancer
    elif "câncer" in text or "carcinoma" in text or "malignancy" in text or "cancer" in text:
        chunk.metadata["topic"] = "thyroid_cancer"

    # Sigmatriol / calcitriol
    elif "calcitriol" in text or "sigmatriol" in text or "vitamina d" in text or "vitamin d" in text:
        chunk.metadata["topic"] = "medication_vitamin_d"

    # Fixare (calcium citrate malate + D3 + K2 + magnesium)
    elif "fixare" in text or "citrato malato" in text or "calcium citrate" in text or "vitamina k2" in text or "vitamin k2" in text:
        chunk.metadata["topic"] = "supplement_fixare"

    # Chlorthalidone
    elif "clortalidona" in text or "chlorthalidone" in text or "diurético" in text or "diuretic" in text:
        chunk.metadata["topic"] = "medication_diuretic"

    # General / unclassified
    else:
        chunk.metadata["topic"] = "general"

In [9]:
import random

# Select random chunks 
chunks_aleatorios = random.sample(chunks, 2)

# Print metadata
for i, chunk in enumerate(chunks_aleatorios, start=1):
    print(f"\n--- Random Chunk {i} ---")
    print("Metadata:")
    print(chunk.metadata)
    print("\nContent (start):")
    print(chunk.page_content[:300])


--- Random Chunk 1 ---
Metadata:
{'producer': 'PDFCreator 2.3.0.103', 'creator': 'PDFCreator 2.3.0.103', 'creationdate': '2019-02-14T14:01:09-02:00', 'moddate': '2019-02-14T14:01:09-02:00', 'title': 'Bula_Clortalidona_Profissional_0219A', 'author': 'mariana.costa', 'subject': '', 'keywords': '', 'source': 'Bula-clortalidona-Profissional', 'total_pages': 13, 'page': 9, 'page_label': '10', 'topic': 'medication_diuretic'}

Content (start):
Este medicamento não deve ser partido ou mastigado.  
 
9. REAÇÕES ADVERSAS  
As reações adversas a seguir, são derivadas de diversas fontes, incluindo experiência pós-comercialização com 
clortalidona, estão listados por classe de sistemas de órgãos do MedDRA. Dentro de cada classe de sistemas de ó

--- Random Chunk 2 ---
Metadata:
{'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20220310182959', 'source': 'bula_sigmatriol', 'total_pages': 9, 'page': 4, 'page_label': '5', 'topic': 'medication_vitamin_d'}

Content (start):
hipercalcemia. 

In [ ]:
# Inicializa o modelo de embeddings (forma atual)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"  # opcional, mas recomendado
)

# Cria o banco vetorial com os chunks
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

In [11]:
# Cria o retriever para busca semântica
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}  # Número de chunks retornados
)

In [14]:
# # Inicializa o modelo de linguagem
# llm = ChatOpenAI(
#     model="gpt-4o-mini",
# )

# # Cria a cadeia RAG
# qa_chain = RetrievalQA.from_chain_type(
#     llm=llm,
#     chain_type="stuff",
#     retriever=retriever,
#     return_source_documents=True
# )

In [15]:
# Prompt customizado
from langchain_core.prompts import PromptTemplate

prompt_template = """Use o contexto abaixo para responder a pergunta.
Os documentos podem estar em inglês ou português — use o conteúdo independente do idioma.
Responda sempre em português, de forma clara e acessível.

Contexto:
{context}

Pergunta: {question}

Resposta:"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

# Integration RAG Pipeline (já existia)
llm = ChatOpenAI(model="gpt-4o-mini")

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}  # ← única linha nova aqui
)

In [17]:
resposta = qa_chain.invoke("Sinto as vezes dores nas mãos, mesmo tomando o calcio, isso pode indicar meu cálcio desequilibrado?")
print(resposta["result"])

Sim, as dores nas mãos podem ser um sinal de um desequilíbrio nos níveis de cálcio no organismo. É importante considerar que a hipocalcemia, que é a deficiência de cálcio, ou a hipercalcemia, que é o excesso, podem causar diversos sintomas, incluindo dores musculares, dormências e cãibras. Caso você esteja sentindo esse tipo de dor, é recomendável que consulte um médico para realizar exames e avaliar seus níveis de cálcio e outras substâncias no sangue. O acompanhamento médico é fundamental para garantir que você esteja recebendo a dose adequada de cálcio e para ajustar o tratamento, se necessário.


In [18]:
from IPython.display import Markdown, display
display(Markdown(resposta["result"]))

Sim, as dores nas mãos podem ser um sinal de um desequilíbrio nos níveis de cálcio no organismo. É importante considerar que a hipocalcemia, que é a deficiência de cálcio, ou a hipercalcemia, que é o excesso, podem causar diversos sintomas, incluindo dores musculares, dormências e cãibras. Caso você esteja sentindo esse tipo de dor, é recomendável que consulte um médico para realizar exames e avaliar seus níveis de cálcio e outras substâncias no sangue. O acompanhamento médico é fundamental para garantir que você esteja recebendo a dose adequada de cálcio e para ajustar o tratamento, se necessário.

In [19]:
print("\n--- Fontes utilizadas ---")
for i, doc in enumerate(resposta["source_documents"], start=1):
    print(f"\nChunk {i}")
    print(f"Arquivo: {doc.metadata.get('source', 'N/A')}")
    print(f"Página:  {doc.metadata.get('page', 'N/A')}")
    print(f"Tema:    {doc.metadata.get('topic', 'N/A')}")
    print(f"Trecho:  {doc.page_content[:200]}")


--- Fontes utilizadas ---

Chunk 1
Arquivo: Bula-Sigmatriol-Profissional-Consulta-Remedios
Página:  6
Tema:    medication_vitamin_d
Trecho:  aos da hipervitaminose D, ou seja: síndrome de hipercalcemia ou intoxicação por cálcio (dependendo da severidade e duração da 
hipercalcemia). Sintomas agudos ocasionais incluem anorexia, cefaléia, vô

Chunk 2
Arquivo: bula_sigmatriol
Página:  6
Tema:    medication_vitamin_d
Trecho:  aos da hipervitaminose D, ou seja: síndrome de hipercalcemia ou intoxicação por cálcio (dependendo da severidade e duração da 
hipercalcemia). Sintomas agudos ocasionais incluem anorexia, cefaléia, vô

Chunk 3
Arquivo: protocolo-clinico-e-diretrizes-terapeuticas-hipoparatireoidismo
Página:  12
Tema:    general
Trecho:  com	aparecimento	de	dormências,	cãibras,	dor	nos	músculos	e	sintomas	mais	graves,	como	
alterações	de	comportamento	e	até	convulsões.
2		MEDICAMENTOS
•	 Estes	medicamentos	não	curam	a	doença,	mas	leva

Chunk 4
Arquivo: protocolo-clinico-e-diretrizes-te